# 23 — ABSA review-level smoke test

Validates the ABSA labeling prompt + JSON parsing on a handful of reviews via the **standard API** (not Batch) before firing the full batch.

Flow: pick a few reviews → build the exact same request `absa_label_submit.py` would send → call `messages.create` → parse, normalize (snake_case), **derive `key_aspect`**, validate evidence substrings → inspect.

**Nothing here writes to the DB** — inspection only. Cost: a few cents on the standard API.

In [1]:
import sys
sys.path.insert(0, "../src")

import duckdb
import json

from absa_label import (
    DB_PATH, MODEL, MAX_TOKENS, SYSTEM_PROMPT, BATCH_OUTPUT_SCHEMA,
    N_REVIEWS_PER_REQUEST, MAX_CHUNK_CHARS,
    build_user_prompt, pack_reviews, parse_batch_result,
    normalize_aspects, rollup, get_client,
)

N_TEST_REVIEWS = 20  # 20-30 for the full validation pass (chunked automatically)

con = duckdb.connect(str(DB_PATH), read_only=True)
print(MODEL, "| max_tokens:", MAX_TOKENS,
      "| per request: <=", N_REVIEWS_PER_REQUEST, "reviews & <=", MAX_CHUNK_CHARS, "chars")

claude-sonnet-5 | max_tokens: 12000 | per request: <= 10 reviews & <= 5000 chars


In [2]:
# Mixed en/vi reviews with some substance (>=120 chars), deterministic pick
rows = con.execute("""
    SELECT review_id, language, review_text
    FROM REVIEW_DATA
    WHERE review_text IS NOT NULL AND length(review_text) BETWEEN 120 AND 800
    ORDER BY md5(review_id)
    LIMIT ?
""", [N_TEST_REVIEWS]).fetchall()

reviews = [(r[0], r[1] or "en", r[2]) for r in rows]
for rid, lang, text in reviews:
    print(f"[{lang}] {rid}: {text[:100]}...")

[en] ChZDSUhNMG9nS0VJTy04LV9JbnU2UUNnEAE: We had a lovely stay at Pullman Vung. Its location was ideal for the tours we had planned. The hotel...
[en] bf2844b5-fdf0-4eaf-9c58-68be4ca6b04c: Wonderful! It is new so a few areas needing attention but such friendly, caring people who made us f...
[vi] ChdDSUhNMG9nS0VJQ0FnSUNLaGViWDRRRRAB: century hotel nằm cạnh dòng sông hương thơ mộng, có vew sông, và ngay trung tâm thành  phố, thuận ti...
[en] ChdDSUhNMG9nS0VJQ0FnSUNJeFktbDBnRRAB: This Hotel is perfectly located to visit Hoi An's old town and night markets. It's just a short walk...
[vi] Ci9DQUlRQUNvZENodHljRjlvT21aalIwWmpZbmM1UjNGaGFtRmpkVU5aU0doNUxYYxAB: Khách sạn đáng đồng tiền, rất thích hợp để nghỉ dưỡng, nhân viên rất tốt bụng và thân thiện, vị trí ...
[vi] ChdDSUhNMG9nS0VJQ0FnTUNvNFAtbWlnRRAB: Về vị trí thì ok vì ra khu vui chơi, bãi biển và khu bar ngoài trời, chợ đêm khá gần, xung quanh nhi...
[vi] ChdDSUhNMG9nS0VJQ0FnTURvOTRhN19BRRAB: Dịch vụ rất thân thiện! Lễ tân có dịch vụ đổ

In [ ]:
# Same chunking as the batch: pack_reviews caps BOTH review count and total
# chars per request. The prompt now uses positional aliases (r1..rN) instead
# of raw review_ids (the model mis-transcribed ~0.5% of the long ids in
# wave 1); aliases are mapped back to real ids per chunk.
from absa_label import alias_map

client = get_client()

results = []
for ci, batch in enumerate(pack_reviews(reviews)):
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_user_prompt(batch)}],
        output_config={"format": {"type": "json_schema", "schema": BATCH_OUTPUT_SCHEMA}},
    )
    print(f"chunk {ci}: {len(batch)} reviews  stop_reason={response.stop_reason}  "
          f"in={response.usage.input_tokens}  out={response.usage.output_tokens}")
    assert response.stop_reason == "end_turn", (
        f"output truncated (stop_reason={response.stop_reason}) - "
        f"raise MAX_TOKENS or lower MAX_CHUNK_CHARS in src/absa_label.py"
    )
    raw = next(b.text for b in response.content if b.type == "text")
    aliases = alias_map([r[0] for r in batch])
    for rv in parse_batch_result(raw):
        rv["review_id"] = aliases.get(rv["review_id"], rv["review_id"])
        results.append(rv)

print(f"\nparsed {len(results)} review results")

In [4]:
# Normalize -> derive key_aspect -> validate evidence (results from all chunks)
texts = {rid: text for rid, _, text in reviews}

n_aspects = n_bad = n_other = 0
for rv in results:
    rid = rv["review_id"]
    aspects = normalize_aspects(rv["aspects"], texts.get(rid, ""))
    print(f"\n=== {rid} ===")
    print(texts.get(rid, "")[:200])
    for a in aspects:
        flag = "" if a["evidence_valid"] else "  << EVIDENCE NOT A SUBSTRING"
        coined = " (coined)" if a["key_aspect"] == "other" else ""
        print(f"  {a['key_aspect']:10} <- {a['sub_aspect']:24}{coined} "
              f"{a['sentiment']:8} | {a['evidence'][:60]!r}{flag}")
        n_aspects += 1
        n_bad += 0 if a["evidence_valid"] else 1
        n_other += 1 if a["key_aspect"] == "other" else 0
    print("  rollup:", {k: v for k, v in rollup(aspects).items() if v})

print(f"\nTotals: {n_aspects} aspects | invalid evidence: {n_bad} | "
      f"key_aspect=other (coined subs): {n_other}")


=== ChZDSUhNMG9nS0VJTy04LV9JbnU2UUNnEAE ===
We had a lovely stay at Pullman Vung. Its location was ideal for the tours we had planned. The hotel is located close to the beach which was having a lot of maintenance/ upgrades when we were there. T
  experience <- overall_satisfaction     positive | 'We had a lovely stay at Pullman Vung'
  amenity    <- transport_access         positive | 'Its location was ideal for the tours we had planned'
  amenity    <- coastal_access           negative | 'close to the beach which was having a lot of maintenance/ up'
  experience <- coastal_view             positive | 'The beachfront area will be really lovely'
  rollup: {'amenity': 'neutral', 'experience': 'positive'}

=== bf2844b5-fdf0-4eaf-9c58-68be4ca6b04c ===
Wonderful! It is new so a few areas needing attention but such friendly, caring people who made us feel like family :) We loved being out of Hoi An city but this may not suit others. Many other hotels
  facility   <- facility_condition      

## Checklist before the full batch

- [ ] Every emitted `review_id` matches an input id
- [ ] `evidence` substrings valid (target ≈ 100%; investigate any failure)
- [ ] Coined sub_aspects (`key_aspect=other`) are rare and sensible
- [ ] vi evidence keeps diacritics verbatim
- [ ] Multi-row aspects appear where reviews mix praise + complaint

If all good:
```bash
uv run python src/absa_sample.py                 # build the 15k sample
uv run python src/absa_label_submit.py --dry-run # inspect
uv run python src/absa_label_submit.py           # submit (YOU run this)
uv run python src/absa_label_retrieve.py --wait  # later
```